<a href="https://colab.research.google.com/github/Maarij-Ahmed-Khan/-Multimodal-ML-Housing-Price-Prediction-Using-Images-/blob/main/Task_3_Multimodal_ML_%E2%80%93_Housing_Price_Prediction_Using_Images_%2B_Tabular_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 3: Multimodal ML – Housing Price Prediction Using Images + Tabular
Data

In [ ]:
# ============================================
# MULTIMODAL HOUSING PRICE PREDICTION
# Using Images + Tabular Data
# ============================================

# INSTALL REQUIRED LIBRARIES
# Uncomment if needed

# !pip install tensorflow pandas numpy matplotlib scikit-learn pillow kagglehub

# ============================================
# STEP 1: IMPORT LIBRARIES
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Flatten,
    Dropout,
    Concatenate
)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# ============================================
# STEP 2: DOWNLOAD DATASET DIRECTLY
# ============================================

# This uses California Housing dataset
# and generates sample images automatically

from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)

data = housing.frame

# ============================================
# STEP 3: CREATE TARGET COLUMN
# ============================================

# House price column

data['Price'] = data['MedHouseVal'] * 100000

# Select useful columns

data = data[[
    'MedInc',
    'HouseAge',
    'AveRooms',
    'AveBedrms',
    'Population',
    'AveOccup',
    'Latitude',
    'Longitude',
    'Price'
]]

print("\nDataset Shape:")
print(data.shape)

print("\nFirst 5 Rows:")
print(data.head())

# ============================================
# STEP 4: CREATE DUMMY HOUSE IMAGES
# ============================================

# Since California Housing dataset has no images,
# we create synthetic images automatically

IMAGE_FOLDER = "house_images"

os.makedirs(IMAGE_FOLDER, exist_ok=True)

IMG_SIZE = 128

print("\nGenerating house images...")

for i in range(len(data)):

    # Create random image
    image = np.random.randint(
        0,
        255,
        (IMG_SIZE, IMG_SIZE, 3),
        dtype=np.uint8
    )

    # Save image
    plt.imsave(
        f"{IMAGE_FOLDER}/house_{i}.jpg",
        image
    )

# Add image names to dataframe

data['image_name'] = [
    f"house_{i}.jpg"
    for i in range(len(data))
]

print("Images generated successfully!")

# ============================================
# STEP 5: LOAD IMAGES
# ============================================

def load_images(image_names):

    images = []

    for name in image_names:

        path = os.path.join(
            IMAGE_FOLDER,
            name
        )

        img = load_img(
            path,
            target_size=(IMG_SIZE, IMG_SIZE)
        )

        img = img_to_array(img)

        img = img / 255.0

        images.append(img)

    return np.array(images)

print("\nLoading images...")

X_images = load_images(data['image_name'])

print("Image Data Shape:")
print(X_images.shape)

# ============================================
# STEP 6: PREPARE TABULAR DATA
# ============================================

X_tabular = data[[
    'MedInc',
    'HouseAge',
    'AveRooms',
    'AveBedrms',
    'Population',
    'AveOccup',
    'Latitude',
    'Longitude'
]]

y = data['Price']

# Normalize features

scaler = StandardScaler()

X_tabular = scaler.fit_transform(X_tabular)

print("\nTabular Data Shape:")
print(X_tabular.shape)

# ============================================
# STEP 7: TRAIN TEST SPLIT
# ============================================

(
    X_img_train,
    X_img_test,
    X_tab_train,
    X_tab_test,
    y_train,
    y_test
) = train_test_split(
    X_images,
    X_tabular,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining Data Ready!")

# ============================================
# STEP 8: BUILD CNN MODEL
# ============================================

# Pretrained CNN

cnn_base = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)

cnn_base.trainable = False

# Image Input

image_input = Input(
    shape=(128, 128, 3)
)

x = cnn_base(image_input)

x = Flatten()(x)

x = Dense(
    128,
    activation='relu'
)(x)

x = Dropout(0.3)(x)

# ============================================
# STEP 9: TABULAR MODEL
# ============================================

tabular_input = Input(
    shape=(8,)
)

t = Dense(
    64,
    activation='relu'
)(tabular_input)

t = Dense(
    32,
    activation='relu'
)(t)

# ============================================
# STEP 10: FEATURE FUSION
# ============================================

combined = Concatenate()([x, t])

z = Dense(
    64,
    activation='relu'
)(combined)

z = Dense(
    32,
    activation='relu'
)(z)

output = Dense(1)(z)

# ============================================
# STEP 11: FINAL MODEL
# ============================================

model = Model(
    inputs=[
        image_input,
        tabular_input
    ],
    outputs=output
)

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

print("\nMODEL SUMMARY:\n")

model.summary()

# ============================================
# STEP 12: TRAIN MODEL
# ============================================

print("\nTraining Model...\n")

history = model.fit(
    [X_img_train, X_tab_train],
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=32
)

# ============================================
# STEP 13: MAKE PREDICTIONS
# ============================================

print("\nMaking Predictions...\n")

predictions = model.predict(
    [X_img_test, X_tab_test]
)

# ============================================
# STEP 14: EVALUATION
# ============================================

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

print("\n===================================")
print("MODEL EVALUATION")
print("===================================")

print(f"MAE  : {mae:.2f}")

print(f"RMSE : {rmse:.2f}")

# ============================================
# STEP 15: PLOT LOSS GRAPH
# ============================================

plt.figure(figsize=(8,5))

plt.plot(
    history.history['loss'],
    label='Training Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.show()

# ============================================
# STEP 16: SAMPLE PREDICTIONS
# ============================================

print("\n===================================")
print("SAMPLE PREDICTIONS")
print("===================================")

for i in range(5):

    actual = y_test.iloc[i]

    predicted = predictions[i][0]

    print(f"\nHouse {i+1}")

    print(f"Actual Price    : {actual:.2f}")

    print(f"Predicted Price : {predicted:.2f}")

# ============================================
# END OF PROJECT
# ============================================

print("\nProject Completed Successfully!")


Dataset Shape:
(20640, 9)

First 5 Rows:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude     Price  
0    -122.23  452600.0  
1    -122.22  358500.0  
2    -122.24  352100.0  
3    -122.25  341300.0  
4    -122.25  342200.0  

Generating house images...
Images generated successfully!

Loading images...
Image Data Shape:
(20640, 128, 128, 3)

Tabular Data Shape:
(20640, 8)

Training Data Ready!

MODEL SUMMARY:



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_1… │ (None, 4, 4,      │  2,257,984 │ input_layer_1[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 20480)     │          0 │ mobilenetv2_1.00… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │  2,621,568 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │        576 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 160)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │     10,304 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      2,080 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         33 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,894,625 (18.67 MB)

 Trainable params: 2,636,641 (10.06 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


Training Model...

